In [22]:
import numpy as np
import pandas as pd

In [23]:
df = pd.read_csv("../data/languages.csv")

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           2722 non-null   int64  
 1   Name in English              2722 non-null   str    
 2   Name in French               2699 non-null   str    
 3   Name in Spanish              2701 non-null   str    
 4   Countries                    2721 non-null   str    
 5   Country codes alpha 3        2721 non-null   str    
 6   ISO639-3 codes               2458 non-null   str    
 7   Degree of endangerment       2722 non-null   str    
 8   Alternate names              1583 non-null   str    
 9   Name in the language         27 non-null     str    
 10  Number of speakers           2539 non-null   float64
 11  Sources                      2079 non-null   str    
 12  Latitude                     2719 non-null   float64
 13  Longitude                    

In [25]:
df[df["Number of speakers"].isna()]["Degree of endangerment"].value_counts()

Degree of endangerment
Critically endangered    63
Severely endangered      49
Definitely endangered    45
Vulnerable               17
Extinct                   9
Name: count, dtype: int64

Existen valores nulos en todas las categorías, lo que hace que pierda peso la teoría inicial de que posiblemente las lenguas que no tengan valor para la variable "Nombre of speakers" sea porque ya están extintas o tengan valor = 0.

In [32]:
df[df["Number of speakers"].isna()]["Countries"].nunique()

65

In [27]:
df[df["Number of speakers"].isna()]["Countries"].value_counts()

Countries
Colombia                                                                                                                                                                                                                                                                                                             20
Canada                                                                                                                                                                                                                                                                                                               19
Russian Federation                                                                                                                                                                                                                                                                                                   12
Cambodia                                              

Las lenguas con valores nulos en "Number of speakers" están presentes en más de 65 países diferentes.

In [28]:
(
    df.groupby('Degree of endangerment')['Number of speakers']
    .apply(lambda x: x.isna().mean())
    .sort_values(ascending=False)
)

Degree of endangerment
Critically endangered    0.103789
Severely endangered      0.088448
Definitely endangered    0.066176
Extinct                  0.035573
Vulnerable               0.027070
Name: Number of speakers, dtype: float64

Calculamos el porcentaje que suponen las filas con valor nulo en la columna "Number of speakers" según el total de cada categoría en cuanto al peligro de extinción de las lenguas:

- Dentro de las lenguas categorizadas como **'Critically endangered'**, hay un **10.37%** de lenguas con valor NaN en la columna "Number of speakers"
- Dentro de las lenguas categorizadas como **'Severely endangered'**, hay un **8.84%** de lenguas con valor NaN en la columna "Number of speakers"
- Dentro de las lenguas categorizadas como **'Definitely endangered'**, hay un **6.61%** de lenguas con valor NaN en la columna "Number of speakers"
- Dentro de las lenguas categorizadas como **'Extinct'**, hay un **3.55%** de lenguas con valor NaN en la columna "Number of speakers"
- Dentro de las lenguas categorizadas como **'Vulnerable'**, hay un **2.7%** de lenguas con valor NaN en la columna "Number of speakers"

---

Hacemos un .explode() temporal para separar los países en diferentes filas en los casos en los que la columna Countries tenga varios países:

In [34]:
df_exploded = df.copy()

df_exploded['Countries'] = (
    df_exploded['Countries']
    .str.split(', ')
)

df_exploded = df_exploded.explode('Countries')

In [36]:
conutries_mas10 = []

df_countries_nan = df_exploded[df_exploded['Number of speakers'].isna()]['Countries'].value_counts()

for country in df_countries_nan.index:
    if  df_countries_nan.loc[country] >= 10:
        conutries_mas10.append(country)
        
conutries_mas10

['Canada',
 'Colombia',
 'Russian Federation',
 'India',
 'United States of America',
 'Cambodia',
 'France']

Creamos una lista que contenga los países que estén presentes en 10 o más lenguas que tengan valores nulos en la variable 'Number of speakers', y llegamos a la conclusión de que todos estos países son:
- Países de gran tamaño 
- Con un número elevado de habitantes
- Muy diversos lingüisticamente
- Con una gran cantidad de lenguas indígenas o minoritarias

In [38]:
(
    df.isna()
    .groupby(df['Number of speakers'].isna())
    .mean()
    .T
)

Number of speakers,False,True
ID,0.000000,0.000000
Name in English,0.000000,0.000000
Name in French,0.008665,0.005464
Name in Spanish,0.007877,0.005464
Countries,0.000394,0.000000
Country codes alpha 3,0.000394,0.000000
ISO639-3 codes,0.090981,0.180328
Degree of endangerment,0.000000,0.000000
Alternate names,0.404884,0.606557
Name in the language,0.991729,0.967213


La columna False nos muestra el porcentaje de valores nulos de cada variable entre las filas que sí tienen Number of speakers.

La columna True nos muestra el porcentaje de valores nulos de cada variable entre las filas que NO tienen Number of speakers, es decir, que en 'Number of speakers' tienen NaN.

Con esta información podemos ver que, entre los registros que tienen NaN en 'Number of speakers', el 47.57% no tiene información asociada a la variable 'Sources', lo que significa que son lenguas poco documentadas sin fuentes asociadas a la información obtenida de cada lengua.

Este patrón se repite en la variable 'Alternate Names', con valores nulos en el 60.65% de registros en esta columna entre los registros con valor NaN en 'Number of speakers'. Ocurre lo mismo en la columna 'Description of the location' (46.99%).

Siguiendo la información obtenida a través de este análisis, podemos concluir que las lenguas con valor nulo en la columna 'Number of speakers' son lenguas poco documentadas y con poca información asociada a las mismas.

Esto nos puede llevar a tomar la decisión de valorar la opción de eliminar dichas lenguas del dataset al tratarse de lenguas sobre las que no abunda la información, y en las que casi la mitad de ellas no tienen fuente oficial asociada a la información que figura en dichas lenguas.

Otra opción es eliminar solamente los registros con valor NaN en 'Number of speakers' que no tengan fuente asociada (Source = NaN), e imputar los valores de esta columna en el resto de registros que si tengan fuente asociada.

In [40]:
len(df[df["Number of speakers"].isna()])/len(df)

0.06722997795738428

Las filas con valor nulo en 'Number of speakers' representan un 6.72% por ciento sobre el total, lo cual supone una gran perdida de información en caso de deshacernos de todas estas filas.